# Notebook 04 Probability Distributions & Statistical Modeling

In the previous notebooks, we explored the geometry of data using linear algebra (vectors, matrices, PCA, and SVD). Here, we introduce the probabilistic perspective. Machine learning models are often formulated as probabilistic models, and learning is framed as finding the parameters that make the observed data most likely.

Our journey:
$$
\boxed{
\text{Probability} \rightarrow \text{Distributions} \rightarrow \text{Conditional Probability} \rightarrow \text{Bayes} \rightarrow \text{Likelihood} \rightarrow \text{MLE} \rightarrow \text{Probabilistic Models}
}
$$

## 1. Probability Foundations Revision

A **random variable** $X$ maps outcomes of a random process to numerical values.

- **Discrete Random Variables**: Described by a Probability Mass Function (PMF), $p(x) = P(X = x)$.
- **Continuous Random Variables**: Described by a Probability Density Function (PDF), $f(x)$. The probability of $X$ falling in an interval is the integral of the PDF over that interval.

The **Cumulative Distribution Function (CDF)** applies to both and gives the probability that $X$ is less than or equal to $x$:
$$ F_X(x) = P(X \le x) $$

For continuous variables (where differentiable):
$$ f_X(x) = \frac{d}{dx}F_X(x) \quad \text{and} \quad F_X(x) = \int_{-\infty}^{x} f_X(t) \, dt $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import ipywidgets as widgets
from IPython.display import display

# Ensure our src directory is in the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.linear_algebra_and_stat.core.probability import gaussian_pdf

In [ ]:
# Visualizing PDF and CDF for a Standard Normal Distribution
from scipy.stats import norm
x_vals = np.linspace(-4, 4, 100)
pdf_vals = norm.pdf(x_vals)
cdf_vals = norm.cdf(x_vals)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(x_vals, pdf_vals, 'b-')
ax[0].set_title('Probability Density Function (PDF)')
ax[0].grid(True, alpha=0.3)

ax[1].plot(x_vals, cdf_vals, 'r-')
ax[1].set_title('Cumulative Distribution Function (CDF)')
ax[1].grid(True, alpha=0.3)
plt.show()

## 2. Expectation, Variance, and Covariance

### Expectation
The expected value $E[X]$ is the probability-weighted average of all possible values.
For discrete variables: $E[X] = \sum_x x p(x)$
For continuous variables: $E[X] = \int_{-\infty}^{\infty} x f(x) \, dx$

**Linearity of Expectation** (holds even if $X$ and $Y$ are NOT independent):
$$ E[aX + b] = aE[X] + b $$
$$ E[X + Y] = E[X] + E[Y] $$

### Variance 
Variance measures the expected squared deviation from the mean, $\mu = E[X]$:
$$ Var(X) = E[(X - E[X])^2] $$

Let's derive the alternative form step-by-step:
First, expand the squared term inside the expectation:
$$ (X - \mu)^2 = X^2 - 2\mu X + \mu^2 $$

Next, take the expectation of this expression. Using the linearity of expectation, we distribute $E[\cdot]$ across the terms:
$$ E[(X - \mu)^2] = E[X^2 - 2\mu X + \mu^2] = E[X^2] - 2\mu E[X] + E[\mu^2] $$

Since $\mu$ is a constant, $E[\mu^2] = \mu^2$. And by definition, $E[X] = \mu$:
$$ E[X^2] - 2\mu(\mu) + \mu^2 = E[X^2] - 2\mu^2 + \mu^2 = E[X^2] - \mu^2 $$

Substituting $\mu = E[X]$ back in, we get the fundamental variance formula:
$$ \boxed{ Var(X) = E[X^2] - (E[X])^2 } $$

Geometrically (from our Linear Algebra notebooks), variance measures the squared "distance" or "spread" from the center of mass.

### Covariance 
Covariance measures how two variables change together:
$$ Cov(X,Y) = E[(X - E[X])(Y - E[Y])] $$

Let's derive the alternative form step-by-step:
First, expand the product:
$$ (X - E[X])(Y - E[Y]) = XY - X E[Y] - Y E[X] + E[X]E[Y] $$

Take the expectation. Remember that $E[X]$ and $E[Y]$ are constants:
$$ E[XY - X E[Y] - Y E[X] + E[X]E[Y]] $$
$$ = E[XY] - E[X]E[Y] - E[Y]E[X] + E[X]E[Y] $$
$$ = E[XY] - E[X]E[Y] $$

$$ \boxed{ Cov(X,Y) = E[XY] - E[X]E[Y] } $$

**Geometric Connection to PCA:**
In the PCA notebook, we defined centered data $x_c = x - E[X]$ and $y_c = y - E[Y]$. Their empirical covariance relates directly to the dot product $x_c^T y_c$.
- $\text{variance} \leftrightarrow \text{self dot product}$
- $\text{covariance} \leftrightarrow \text{cross dot product}$

## 3. Important Distributions

### Bernoulli Distribution
A Bernoulli trial has two outcomes: 1 (success) with probability $p$, or 0 (failure) with probability $1-p$.
$$ X \sim Bernoulli(p) $$
$$ P(X=x) = p^x (1-p)^{1-x} \quad \text{for } x \in \{0, 1\} $$
This compact exponent formula works beautifully because if $x=1$, we get $p^1 (1-p)^0 = p$. If $x=0$, we get $p^0 (1-p)^1 = 1-p$.

**Deriving Expectation:**
$$ E[X] = \sum_{x \in \{0, 1\}} x P(X=x) = (0 \cdot (1-p)) + (1 \cdot p) = p $$

**Deriving Variance:**
$$ E[X^2] = (0^2 \cdot (1-p)) + (1^2 \cdot p) = p $$
$$ Var(X) = E[X^2] - E[X]^2 = p - p^2 = p(1-p) $$

In ML, Bernoulli distributions model binary classification where $Y \in \{0, 1\}$ and the model predicts $P(Y=1 | X) = p$.

### Binomial Distribution
Models the number of successes $k$ in $n$ independent Bernoulli trials.
$$ P(X=k) = \binom{n}{k} p^k (1-p)^{n-k} $$
Where $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ counts the number of different ways to arrange $k$ successes in $n$ trials. 

Since a Binomial random variable is the sum of $n$ independent Bernoulli variables ($X = X_1 + X_2 + \dots + X_n$):
$$ E[X] = \sum_{i=1}^n E[X_i] = np $$
$$ Var(X) = \sum_{i=1}^n Var(X_i) = np(1-p) $$

### Gaussian (Normal) Distribution
The most important continuous distribution, completely defined by its mean $\mu$ (location) and variance $\sigma^2$ (spread).
$$ X \sim N(\mu, \sigma^2) $$
$$f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

Let's break this down:
- The exponent $-\frac{(x-\mu)^2}{2\sigma^2}$ creates the symmetric "bell" shape. The $(x-\mu)^2$ part penalizes values far from the mean, and dividing by $2\sigma^2$ controls how wide the bell is.
- The constant $\frac{1}{\sqrt{2\pi\sigma^2}}$ ensures the total area under the curve integrates exactly to 1, a strict requirement for a valid probability distribution.

**Standard Normal Transformation:**
Any Gaussian can be transformed into a Standard Normal $N(0, 1)$ by standardizing:
$$ Z = \frac{X - \mu}{\sigma} $$

In [ ]:
# Visualizing Gaussian distributions
x_vals = np.linspace(-10, 10, 400)

plt.figure(figsize=(8, 5))
plt.plot(x_vals, gaussian_pdf(x_vals, 0, 1), label='$\\mu=0, \\sigma^2=1$ (Standard)')
plt.plot(x_vals, gaussian_pdf(x_vals, 0, 2), label='$\\mu=0, \\sigma^2=4$ (Wider)')
plt.plot(x_vals, gaussian_pdf(x_vals, -3, 1), label='$\\mu=-3, \\sigma^2=1$ (Shifted)')
plt.title("Gaussian Distributions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Experiment A: Law of Large Numbers (LLN)

The LLN states that the sample average converges to the true expectation as the sample size $n$ increases.
$$ \bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \rightarrow E[X] $$

> **Crucial distinction:** The LLN concerns the convergence of *sample averages*, it does not mean that every individual sample eventually becomes close to the expectation.

Let's run a simulation.

In [ ]:
from src.linear_algebra_and_stat.assignments.lln_experiment import run_lln_experiment

# Simulating Bernoulli trials with p=0.7
run_lln_experiment(n_samples=5000, p=0.7)

## 5. Conditional Probability & Bayes' Theorem

### Conditional Probability
$$ P(A \mid B) = \frac{P(A \cap B)}{P(B)} \quad \text{where } P(B) > 0 $$
Multiplication rule:
$$ P(A \cap B) = P(A \mid B)P(B) = P(B \mid A)P(A) $$

### Independence
If $A$ and $B$ are independent:
$$ P(A \cap B) = P(A)P(B) $$
This implies $P(A \mid B) = P(A)$.

> **Crucial Note:** Independence is an assumption that allows joint probabilities to factorize. For independent observations $X_1, \dots, X_n$ given $\theta$:
$$ P(X_1, \dots, X_n \mid \theta) = \prod_{i=1}^n P(X_i \mid \theta) $$
This product form is not because "likelihood" inherently means multiplication; it comes entirely from the assumption of independence.

### Bayes' Theorem
From the multiplication rule:
$$ P(A \mid B)P(B) = P(B \mid A)P(A) $$
Divide by $P(B)$:
$$ \boxed{ P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)} } $$

- $P(A)$: **Prior**
- $P(B \mid A)$: **Likelihood**
- $P(B)$: **Evidence**
- $P(A \mid B)$: **Posterior**

Bayes tells us: *What do I believe about $A$ after observing $B$?*
While $P(B \mid A)$ tells us: *How compatible is observation $B$ with $A$ being true?*
These are definitively not interchangeable.

### Monty Hall Worked Derivation
**Scenario 1: Monty knows where the goats are and deliberately opens a goat door.**
Let $C_1$ be the event "car is behind door 1" (your choice). $P(C_1) = 1/3$.
Let $M_2$ be "Monty opens door 2".
If the car is behind door 1, Monty can open 2 or 3. $P(M_2 \mid C_1) = 1/2$.
If the car is behind door 2, Monty CANNOT open door 2. $P(M_2 \mid C_2) = 0$.
If the car is behind door 3, Monty MUST open door 2. $P(M_2 \mid C_3) = 1$.

By the law of total probability, $P(M_2) = P(M_2|C_1)P(C_1) + P(M_2|C_2)P(C_2) + P(M_2|C_3)P(C_3) = (1/2)(1/3) + (0)(1/3) + (1)(1/3) = 1/6 + 1/3 = 1/2$.

Posterior of winning if you stay ($C_1$): $P(C_1 \mid M_2) = \frac{P(M_2 \mid C_1)P(C_1)}{P(M_2)} = \frac{(1/2)(1/3)}{1/2} = 1/3$.
Posterior of winning if you switch ($C_3$): $P(C_3 \mid M_2) = \frac{P(M_2 \mid C_3)P(C_3)}{P(M_2)} = \frac{(1)(1/3)}{1/2} = 2/3$.

**Scenario 2: Monty randomly opens a door, and it happens to be a goat.**
Now, $P(M_2 \mid C_1) = 1/2$, $P(M_2 \mid C_2) = 1/2$, and $P(M_2 \mid C_3) = 1/2$.
$P(M_2) = 1/2$.
Posterior of winning if you switch ($C_3$): $P(C_3 \mid M_2) = \frac{P(M_2 \mid C_3)P(C_3)}{P(M_2)} = \frac{(1/2)(1/3)}{1/2} = 1/3$.
Switching offers no advantage here! The conditional probability model critically depends on the **process that generated the evidence**.

## 6. Bayesian Classification & Naive Bayes

For classes $C_k$, we want the most probable class given observation $X$:
$$ P(C_k \mid X) = \frac{P(X \mid C_k)P(C_k)}{P(X)} $$

Since $P(X)$ is constant across all candidate classes $k$, removing the denominator does not change which class has the largest posterior. Thus, we can use the "argmax trick":
$$ \hat{C} = \arg\max_k P(X \mid C_k)P(C_k) $$

### How the Calculation Works on Image Pixels
In the upcoming experiment, we classify 8x8 images of digits (64 pixels). Here is exactly how Naive Bayes handles this:

1. **Gaussian Distribution for Each Pixel**: During training, for a specific class (e.g., '0'), the model looks at the pixel in the top-left corner across all images of '0's. It calculates the mean intensity and the variance of that single pixel. It repeats this for all 64 pixels, for all 10 classes.
2. **The Independence Assumption**: We assume that every pixel's intensity is *completely unrelated* to any other pixel's intensity, given the digit class. While obviously false (pixels in strokes are highly correlated), this "naive" assumption vastly simplifies the math.
3. **The Product of 64 Probabilities**: To find the probability $P(X \mid C_k)$ of an entire image, we simply multiply the probabilities of all 64 individual pixels together: $\prod_{j=1}^{64} P(X_j \mid C_k)$.
4. **Log-Likelihood to Prevent Underflow**: Multiplying 64 tiny probabilities results in a number so small a computer rounds it to exactly zero (underflow). To fix this, we take the logarithm, turning the massive product into a sum: $\log \prod p = \sum \log p$.
5. **Log Posterior**: Finally, we add the Log Prior (the baseline probability of the class existing based on our training data) to the Log Likelihood. The class with the highest Log Posterior is our prediction!

### Visualizing the Model's Brain (Bias & Distribution)
Let's see what the model actually learned. The function below plots the mean image and variance image for each digit, and prints the distribution of the training data to check for bias.

In [ ]:
# Plot the learned Gaussian parameters per pixel
from src.linear_algebra_and_stat.assignments.naive_bayes_experiment import visualize_naive_bayes_features

visualize_naive_bayes_features()

### A Major Limitation: Strict Spatial Dependence
Because the model calculates a separate Gaussian for *each fixed pixel coordinate*, it has absolutely zero concept of spatial relationships or shapes. If you draw a perfect '7', but shift it one pixel to the right, the active pixels will land on coordinates where the model expects emptiness (low variance). The likelihood will plummet, and the model will fail entirely! Modern Convolutional Neural Networks (CNNs) were invented specifically to solve this lack of translation invariance.

### Interactive Explorer
Let's test the model. Use the toggle to explore the samples it got right vs. the ones it got wrong. Note how the Log Posterior is visualized: the values are usually large negative numbers, so the score closest to 0 is the winner.

In [ ]:
# Experiment B1: Naive Bayes Classification (Interactive Explorer)
from src.linear_algebra_and_stat.assignments.naive_bayes_experiment import interactive_naive_bayes_experiment

interactive_naive_bayes_experiment()

### Naive Bayes Drawing Canvas
Let's take it a step further! Below is an interactive 8x8 "canvas". You can click the squares to draw your own digits. 
Each click increases the darkness of the pixel (0 -> 4 -> 8 -> 12 -> 16).
When you click "Predict Digit", the Naive Bayes model will calculate the probabilities and tell you what it thinks you drew.


In [ ]:
# Experiment B2: Draw your own digit and test the Naive Bayes Classifier!
from src.linear_algebra_and_stat.assignments.naive_bayes_experiment import interactive_canvas_naive_bayes_experiment

interactive_canvas_naive_bayes_experiment()

## 7. Probability vs Likelihood

Consider the expression $P(X \mid \theta)$. The mathematical expression is exactly the same, but the question being asked changes fundamentally based on what is fixed and what varies.

| Fixed      | Variable   | Interpretation                               |
| ---------- | ---------- | -------------------------------------------- |
| $\theta$ | $X$      | **Probability distribution** over possible observations. "Given a coin with p=0.6, what is the probability of 3 heads in a row?" |
| $x$      | $\theta$ | **Likelihood function** of parameters given observed data. "Given we observed 3 heads in a row, what is the likelihood that p=0.6?" |

$$ L(\theta \mid x) = P(x \mid \theta) $$


## 8. Maximum Likelihood Estimation (MLE)

$$ \boxed{ \hat{\theta}_{MLE} = \arg\max_\theta L(\theta \mid X) } $$
"Choose the parameter value under which the observed data would be most compatible/likely."

Because the logarithm is strictly increasing:
$$ \arg\max_\theta L(\theta) = \arg\max_\theta \log L(\theta) $$
And maximizing $\log L$ is equivalent to minimizing $-\log L$:
$$ \boxed{ \min_\theta -\log L(\theta) } $$
Minimizing Negative Log-Likelihood (NLL) is a fundamental optimization objective in ML.

### Deriving Gaussian MLE using Calculus
For independent observations $x_1, \dots, x_n \sim N(\mu, \sigma^2)$:
$$ L(\mu, \sigma^2) = \prod_{i=1}^n \frac{1}{\sqrt{2\pi\sigma^2}} \exp \left( -\frac{(x_i - \mu)^2}{2\sigma^2} \right) $$

Take the log to convert the product to a sum:
$$ \ell(\mu, \sigma^2) = \sum_{i=1}^n \left( -\frac{1}{2}\log(2\pi\sigma^2) - \frac{(x_i - \mu)^2}{2\sigma^2} \right) $$

**1. Find MLE for $\mu$**:
Differentiate $\ell$ with respect to $\mu$:
$$ \frac{\partial \ell}{\partial \mu} = \sum_{i=1}^n \frac{2(x_i - \mu)}{2\sigma^2} = \frac{1}{\sigma^2} \sum_{i=1}^n (x_i - \mu) $$
Set the derivative to zero:
$$ \sum_{i=1}^n x_i - n\mu = 0 $$
$$ \boxed{ \hat{\mu}_{MLE} = \frac{1}{n}\sum_{i=1}^n x_i = \bar{x} } $$

**2. Find MLE for $\sigma^2$**:
Differentiate $\ell$ with respect to $\sigma^2$:
$$ \frac{\partial \ell}{\partial (\sigma^2)} = \sum_{i=1}^n \left( -\frac{1}{2\sigma^2} + \frac{(x_i - \mu)^2}{2(\sigma^2)^2} \right) $$
Set to zero and solve:
$$ -\frac{n}{2\sigma^2} + \frac{1}{2(\sigma^2)^2}\sum_{i=1}^n (x_i - \mu)^2 = 0 $$
$$ \frac{n}{2\sigma^2} = \frac{1}{2(\sigma^2)^2}\sum_{i=1}^n (x_i - \mu)^2 $$
Multiply by $2(\sigma^2)^2 / n$:
$$ \boxed{ \hat{\sigma}^2_{MLE} = \frac{1}{n} \sum_{i=1}^n (x_i - \hat{\mu})^2 } $$

Notice this is divided by $n$, not $n-1$. The MLE for variance is biased; it underestimates the true variance because we had to estimate $\mu$ from the same data, costing us one "degree of freedom". The unbiased sample variance divides by $n-1$ to correct for this.

In [ ]:
# Experiment C: Maximum Likelihood Laboratory (Interactive)
from src.linear_algebra_and_stat.assignments.mle_experiment import interactive_mle_experiment

interactive_mle_experiment()

### Deriving Bernoulli MLE
For independent observations $y_1, \dots, y_n \in \{0, 1\}$ from $Bernoulli(p)$:
$$ L(p) = \prod_{i=1}^n p^{y_i} (1-p)^{1-y_i} $$
Take the log:
$$ \ell(p) = \sum_{i=1}^n \left( y_i \log p + (1-y_i) \log(1-p) \right) $$
Differentiate with respect to $p$ and set to zero:
$$ \frac{\partial \ell}{\partial p} = \sum_{i=1}^n \left( \frac{y_i}{p} - \frac{1-y_i}{1-p} \right) = 0 $$
$$ \frac{1}{p}\sum_{i=1}^n y_i = \frac{1}{1-p}\sum_{i=1}^n (1-y_i) $$
Let $S = \sum_{i=1}^n y_i$. Then $\sum_{i=1}^n (1-y_i) = n - S$.
$$ \frac{S}{p} = \frac{n-S}{1-p} \implies S(1-p) = p(n-S) \implies S - Sp = pn - Sp \implies S = pn $$
$$ \boxed{ \hat{p}_{MLE} = \frac{S}{n} = \frac{1}{n}\sum_{i=1}^n y_i } $$

The MLE for $p$ is just the sample proportion of 1s!

## 9. Connections to ML Loss Functions (Deep Dive)

One of the most beautiful connections in machine learning is how standard loss functions (like Mean Squared Error and Binary Cross Entropy) are not just arbitrary formulas we invented because they "work well". They are mathematical inevitabilities derived directly from Maximum Likelihood Estimation (MLE) under specific probabilistic assumptions.

Let's elaborate on exactly what this means with concrete examples.

### 1. Bernoulli $\rightarrow$ Binary Cross Entropy (BCE)

**The Setup:**
Imagine a model trying to predict whether a single student passes ($y=1$) or fails ($y=0$) a test based on their hours studied. The model outputs a single number $p$, representing its confidence (probability) that the student passes. For example, $p = 0.8$.

**The Likelihood:**
What is the probability that the model's prediction actually matches reality?
- If the student *passed* ($y=1$), the probability the model assigned to this true event was $p = 0.8$.
- If the student *failed* ($y=0$), the probability the model assigned to this true event was $(1-p) = 0.2$.

We can combine both cases into a single clever formula using exponents:
$$ P(Y=y \mid p) = p^y (1-p)^{1-y} $$
*(Check it: If $y=1$, the $(1-p)$ term becomes $(1-p)^0=1$, leaving just $p$.)*

**Taking the Negative Log:**
To optimize this, we take the negative logarithm (Negative Log-Likelihood):
$$ -\log P(Y=y \mid p) = - \left[ y \log p + (1-y) \log(1-p) \right] $$

**The Result:**
This exact formula is **Binary Cross Entropy (BCE)**! When you train a neural network using BCE, you are not just "punishing it for being wrong". You are explicitly telling the network: *"Assume the output is the parameter $p$ of a Bernoulli coin flip. Now, find the weights that maximize the likelihood of the dataset under that assumption."*

---

### 2. Gaussian Noise $\rightarrow$ Mean Squared Error (MSE)

**The Setup:**
Imagine a model predicting a continuous value, like the price of a house. The model outputs a single prediction $\hat{y} = f_\theta(x)$. However, we know models aren't perfect; there is always random noise in the real world.

**The Gaussian Assumption:**
We assume that the *true* price $y$ is normally distributed (Gaussian) around our model's prediction $\hat{y}$, with some variance $\sigma^2$. 
$$ y \sim N(\hat{y}, \sigma^2) $$

**The Likelihood:**
According to the Gaussian PDF, the likelihood of observing the true price $y$ given our prediction $\hat{y}$ is:
$$ L = \frac{1}{\sqrt{2\pi\sigma^2}} \exp \left( -\frac{(y - \hat{y})^2}{2\sigma^2} \right) $$

If $y$ is very close to $\hat{y}$, the $(y - \hat{y})^2$ term is near $0$, $e^0 = 1$, and the likelihood is high (good!). If $y$ is far away, the exponent becomes a large negative number, $e^{-\text{large}} \approx 0$, and the likelihood plummets (bad!).

**Taking the Negative Log:**
Let's maximize this likelihood by minimizing the negative log-likelihood:
$$ -\log L = - \left[ \log\left(\frac{1}{\sqrt{2\pi\sigma^2}}\right) - \frac{(y - \hat{y})^2}{2\sigma^2} \right] $$

Notice something incredible? The first term $\log(\frac{1}{\dots})$ is just a constant (it doesn't depend on our model's predictions). The denominator $2\sigma^2$ is also just a constant scaling factor. When we run an optimizer (like Gradient Descent) to find the minimum, we can entirely ignore constants!

We are left with minimizing exactly one term:
$$ \arg\min_\theta \sum_{i=1}^n (y_i - \hat{y}_i)^2 $$

**The Result:**
This is exactly **Mean Squared Error (MSE)**! Therefore, whenever you use MSE to train a regression model, you are fundamentally making the probabilistic assumption that *the errors (residuals) of your model are normally distributed (Gaussian)*. 

$$ \boxed{ \text{Gaussian noise assumption} \rightarrow \text{MLE} \rightarrow \text{Squared-error objective} } $$

## 10. Multivariate Gaussian

For a vector $X \in \mathbb{R}^d$, $X \sim N(\mu, \Sigma)$:
$$ f(x) = \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}} \exp \left( -\frac{1}{2} (x-\mu)^T \Sigma^{-1} (x-\mu) \right) $$

$\mu \in \mathbb{R}^d$ is the mean vector, and $\Sigma \in \mathbb{R}^{d \times d}$ is the covariance matrix.
- Diagonal elements are the individual variances $\sigma_i^2$.
- Off-diagonal elements are the covariances $Cov(X_i, X_j)$.

### Mahalanobis Distance
The term inside the exponent is a quadratic form known as the Mahalanobis distance squared:
$$ \boxed{ d_M^2(x, \mu) = (x-\mu)^T \Sigma^{-1} (x-\mu) } $$
Why not use ordinary Euclidean distance $||x-\mu||^2$? Because Euclidean distance assumes the spread of data is isotropic (spherical) and all features are uncorrelated and on the same scale. The covariance matrix $\Sigma$ effectively changes the geometry of distance, scaling and rotating the space to match the shape of the data cloud.
If variables are independent (diagonal $\Sigma$), this simplifies to $\sum_i \frac{(x_i-\mu_i)^2}{\sigma_i^2}$, which is just standardized Euclidean distance.

In [ ]:
# Interactive Multivariate Gaussian Visualization
from src.linear_algebra_and_stat.assignments.gaussian_contours import interactive_multivariate_gaussians

interactive_multivariate_gaussians()